In [1]:
import numpy as np
import sys

def euclidean_distance(a, b):
    # a: [n_samples, n_features], b: [k, n_features]
    # Возвращаем матрицу расстояний [k, n_samples]
    return np.sqrt(np.sum((a[:, np.newaxis, :] - b[np.newaxis, :, :]) ** 2, axis=2)).T

def manhattan_distance(a, b):
    # a: [n_samples, n_features], b: [k, n_features]
    return np.sum(np.abs(a[:, np.newaxis, :] - b[np.newaxis, :, :]), axis=2).T

def cosine_distance(a, b):
    # a: [n_samples, n_features], b: [k, n_features]
    # Косинусное расстояние = 1 - косинусное сходство
    a_norm = a / np.linalg.norm(a, axis=1, keepdims=True)
    b_norm = b / np.linalg.norm(b, axis=1, keepdims=True)
    # Защита от деления на ноль
    a_norm = np.nan_to_num(a_norm, nan=0.0)
    b_norm = np.nan_to_num(b_norm, nan=0.0)
    similarity = np.dot(a_norm, b_norm.T)
    return (1 - similarity).T

def kmeans(X, k, metric='euclidean', max_iters=100, e=1e-4, seed=42):
    np.random.seed(seed)
    n_samples = X.shape[0]
    
    # Инициализация центроидов: случайный выбор k точек из данных
    indices = np.random.choice(n_samples, k, replace=False)
    centroids = X[indices].copy()
    
    if metric == 'euclidean':
        dist_func = euclidean_distance
    elif metric == 'manhattan':
        dist_func = manhattan_distance
    elif metric == 'cosine':
        dist_func = cosine_distance
    else:
        raise ValueError("Unknown metric")
    
    for iteration in range(max_iters):
        # Вычисляем расстояния от всех точек до всех центроидов
        # distances имеет форму [k, n_samples]
        distances = dist_func(X, centroids)
        
        # Назначаем точки ближайшим центроидам
        labels = np.argmin(distances, axis=0)
        
        # Обновляем центроиды
        new_centroids = np.zeros_like(centroids)
        for i in range(k):
            cluster_points = X[labels == i]
            if len(cluster_points) > 0:
                new_centroids[i] = np.mean(cluster_points, axis=0)
            else:
                # Если кластер пуст, оставляем старый центроид
                new_centroids[i] = centroids[i]
        
        # Проверка сходимости
        shift = np.sum((new_centroids - centroids) ** 2)
        if shift < e:
            break
            
        centroids = new_centroids
    
    return centroids, labels

# Считывание данных
def main():
    # Читаем все строки из stdin
    data = sys.stdin.read().strip().split()
    if not data:
        return
        
    # Первая строка: k и metric
    k = int(data[0])
    metric = data[1]
    
    # Остальные данные — координаты точек
    points = []
    for i in range(2, len(data), 2):
        x = float(data[i])
        y = float(data[i+1])
        points.append([x, y])
    
    X = np.array(points)
    
    # Запуск K-Means
    centroids, labels = kmeans(X, k, metric=metric)
    
    # Сортировка центроидов по x-координате
    sort_idx = np.argsort(centroids[:, 0])
    centroids = centroids[sort_idx]
    
    # Вывод результатов с точностью 3 знака после запятой
    for centroid in centroids:
        print(f"{centroid[0]:.3f} {centroid[1]:.3f}")

if __name__ == "__main__":
    main()

KeyboardInterrupt: 